# Public Trade Channel - Real-time Trade Data Analysis

This notebook demonstrates how to connect to the Public Trade Channel WebSocket API to receive real-time trade data and convert it into candlestick format for technical analysis.

## Features
- **Real-time Trade Data**: WebSocket connection to receive live trade executions
- **Candlestick Generation**: Convert trade data to OHLCV format
- **Technical Analysis**: Moving averages, RSI, MACD indicators
- **Visualization**: Interactive candlestick charts and volume analysis
- **Data Persistence**: SQLite database for historical data storage

## API Overview
- **Channel**: `trade`
- **Product Types**: USDT-FUTURES, COIN-FUTURES, USDC-FUTURES
- **Data Format**: Real-time trade executions with price, size, side, timestamp

In [ ]:
# Import required libraries
import websocket
import json
import pandas as pd
import numpy as np
import threading
import time
from datetime import datetime, timedelta
import sqlite3
import logging
from typing import List, Dict, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Technical analysis
try:
    import talib
    TALIB_AVAILABLE = True
except ImportError:
    TALIB_AVAILABLE = False
    print("TA-Lib not available. Some technical indicators will use custom implementations.")

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set style for matplotlib
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
class TradeChannelClient:
    """Enhanced WebSocket client for Public Trade Channel with candlestick support"""
    
    def __init__(self, url: str = "wss://ws-api.bingx.com/market"):
        self.url = url
        self.ws = None
        self.trade_data = []
        self.is_connected = False
        self.subscriptions = {}
        self.candlestick_data = {}  # Store OHLCV data per instrument
        self.last_candle_time = {}  # Track last candle timestamp per instrument
        
    def on_message(self, ws, message):
        """Handle incoming WebSocket messages"""
        try:
            data = json.loads(message)
            
            # Handle subscription confirmation
            if data.get('event') == 'subscribe':
                logger.info(f"Subscribed to {data.get('arg', {}).get('instId')}")
                return
            
            # Handle trade data
            if data.get('action') == 'snapshot' and data.get('arg', {}).get('channel') == 'trade':
                self.process_trade_data(data)
                
        except json.JSONDecodeError as e:
            logger.error(f"JSON decode error: {e}")
        except Exception as e:
            logger.error(f"Error processing message: {e}")
    
    def process_trade_data(self, data: Dict[str, Any]):
        """Enhanced trade data processing with candlestick generation"""
        arg = data.get('arg', {})
        trades = data.get('data', [])
        inst_id = arg.get('instId')
        
        for trade in trades:
            trade_record = {
                'instType': arg.get('instType'),
                'instId': inst_id,
                'timestamp': int(trade.get('ts')),
                'datetime': datetime.fromtimestamp(int(trade.get('ts')) / 1000),
                'price': float(trade.get('price')),
                'size': float(trade.get('size')),
                'side': trade.get('side'),
                'tradeId': trade.get('tradeId'),
                'received_at': datetime.now()
            }
            self.trade_data.append(trade_record)
            
            # Update candlestick data
            self._update_candlestick(trade_record)
            
        logger.info(f"Processed {len(trades)} trades for {inst_id}")
    
    def _update_candlestick(self, trade: Dict[str, Any], timeframe: str = '1min'):
        """Update candlestick data with new trade"""
        inst_id = trade['instId']
        trade_time = trade['datetime']
        price = trade['price']
        volume = trade['size']
        
        # Round to timeframe (1 minute default)
        if timeframe == '1min':
            candle_time = trade_time.replace(second=0, microsecond=0)
        elif timeframe == '5min':
            candle_time = trade_time.replace(minute=(trade_time.minute // 5) * 5, second=0, microsecond=0)
        elif timeframe == '1hour':
            candle_time = trade_time.replace(minute=0, second=0, microsecond=0)
        else:
            candle_time = trade_time.replace(second=0, microsecond=0)
        
        # Initialize candlestick data for instrument if not exists
        if inst_id not in self.candlestick_data:
            self.candlestick_data[inst_id] = []
        
        candles = self.candlestick_data[inst_id]
        
        # Check if we need to create a new candle or update existing one
        if not candles or candles[-1]['timestamp'] != candle_time:
            # Create new candle
            new_candle = {
                'timestamp': candle_time,
                'open': price,
                'high': price,
                'low': price,
                'close': price,
                'volume': volume,
                'trade_count': 1
            }
            candles.append(new_candle)
        else:
            # Update existing candle
            current_candle = candles[-1]
            current_candle['high'] = max(current_candle['high'], price)
            current_candle['low'] = min(current_candle['low'], price)
            current_candle['close'] = price
            current_candle['volume'] += volume
            current_candle['trade_count'] += 1
    
    def on_error(self, ws, error):
        """Handle WebSocket errors"""
        logger.error(f"WebSocket error: {error}")
    
    def on_close(self, ws, close_status_code, close_msg):
        """Handle WebSocket close"""
        self.is_connected = False
        logger.info("WebSocket connection closed")
    
    def on_open(self, ws):
        """Handle WebSocket open"""
        self.is_connected = True
        logger.info("WebSocket connection opened")
    
    def connect(self):
        """Establish WebSocket connection"""
        websocket.enableTrace(True)
        self.ws = websocket.WebSocketApp(
            self.url,
            on_open=self.on_open,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close
        )
        
        # Start connection in separate thread
        self.ws_thread = threading.Thread(target=self.ws.run_forever)
        self.ws_thread.daemon = True
        self.ws_thread.start()
        
        # Wait for connection
        time.sleep(2)
        
    def subscribe(self, inst_type: str, inst_id: str):
        """Subscribe to trade channel for specific instrument"""
        if not self.is_connected:
            logger.error("WebSocket not connected")
            return False
            
        subscription = {
            "op": "subscribe",
            "args": [
                {
                    "instType": "SPOT",  # Thay đổi từ USDT-FUTURES thành SPOT
                    "channel": "trade",
                    "instId": inst_id
                }
            ]
        }
        
        self.ws.send(json.dumps(subscription))
        self.subscriptions[inst_id] = subscription
        logger.info(f"Sent subscription for SPOT {inst_id}")
        return True
    
    def unsubscribe(self, inst_type: str, inst_id: str):
        """Unsubscribe from trade channel"""
        if not self.is_connected:
            logger.error("WebSocket not connected")
            return False
            
        unsubscription = {
            "op": "unsubscribe",
            "args": [
                {
                    "instType": "SPOT",  # Thay đổi từ USDT-FUTURES thành SPOT
                    "channel": "trade",
                    "instId": inst_id
                }
            ]
        }
        
        self.ws.send(json.dumps(unsubscription))
        if inst_id in self.subscriptions:
            del self.subscriptions[inst_id]
        logger.info(f"Unsubscribed from SPOT {inst_id}")
        return True
    
    def get_trade_data(self) -> pd.DataFrame:
        """Get collected trade data as DataFrame"""
        if not self.trade_data:
            return pd.DataFrame()
            
        df = pd.DataFrame(self.trade_data)
        df['datetime'] = pd.to_datetime(df['datetime'])
        return df.sort_values('timestamp')
    
    def get_candlestick_data(self, inst_id: str = None) -> pd.DataFrame:
        """Get candlestick data as DataFrame"""
        if inst_id and inst_id in self.candlestick_data:
            candles = self.candlestick_data[inst_id]
        else:
            # Combine all instruments
            candles = []
            for inst, data in self.candlestick_data.items():
                for candle in data:
                    candle_copy = candle.copy()
                    candle_copy['instId'] = inst
                    candles.append(candle_copy)
        
        if not candles:
            return pd.DataFrame()
        
        df = pd.DataFrame(candles)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df.sort_values('timestamp').reset_index(drop=True)
    
    def close(self):
        """Close WebSocket connection"""
        if self.ws:
            self.ws.close()
        self.is_connected = False

In [ ]:
class CandlestickAnalyzer:
    """Technical analysis for candlestick data"""
    
    def __init__(self, data: pd.DataFrame):
        self.data = data.copy()
        if not self.data.empty:
            self.data = self.data.sort_values('timestamp').reset_index(drop=True)
    
    def add_technical_indicators(self):
        """Add common technical indicators to the data"""
        if self.data.empty:
            return self
        
        # Moving Averages
        self.data['MA5'] = self.data['close'].rolling(window=5).mean()
        self.data['MA10'] = self.data['close'].rolling(window=10).mean()
        self.data['MA20'] = self.data['close'].rolling(window=20).mean()
        
        # Exponential Moving Averages
        self.data['EMA12'] = self.data['close'].ewm(span=12).mean()
        self.data['EMA26'] = self.data['close'].ewm(span=26).mean()
        
        # MACD
        self.data['MACD'] = self.data['EMA12'] - self.data['EMA26']
        self.data['MACD_signal'] = self.data['MACD'].ewm(span=9).mean()
        self.data['MACD_histogram'] = self.data['MACD'] - self.data['MACD_signal']
        
        # RSI
        self.data['RSI'] = self._calculate_rsi(self.data['close'])
        
        # Bollinger Bands
        bb_data = self._calculate_bollinger_bands(self.data['close'])
        self.data = pd.concat([self.data, bb_data], axis=1)
        
        # Volume indicators
        self.data['Volume_MA'] = self.data['volume'].rolling(window=10).mean()
        self.data['Volume_ratio'] = self.data['volume'] / self.data['Volume_MA']
        
        return self
    
    def _calculate_rsi(self, prices: pd.Series, period: int = 14) -> pd.Series:
        """Calculate RSI indicator"""
        if TALIB_AVAILABLE:
            return pd.Series(talib.RSI(prices.values, timeperiod=period), index=prices.index)
        else:
            # Custom RSI calculation
            delta = prices.diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
            rs = gain / loss
            return 100 - (100 / (1 + rs))
    
    def _calculate_bollinger_bands(self, prices: pd.Series, period: int = 20, std_dev: int = 2) -> pd.DataFrame:
        """Calculate Bollinger Bands"""
        sma = prices.rolling(window=period).mean()
        std = prices.rolling(window=period).std()
        
        return pd.DataFrame({
            'BB_upper': sma + (std * std_dev),
            'BB_middle': sma,
            'BB_lower': sma - (std * std_dev)
        })
    
    def detect_patterns(self) -> Dict[str, List[int]]:
        """Detect common candlestick patterns"""
        patterns = {
            'doji': [],
            'hammer': [],
            'shooting_star': [],
            'engulfing_bullish': [],
            'engulfing_bearish': []
        }
        
        if len(self.data) < 2:
            return patterns
        
        for i in range(1, len(self.data)):
            current = self.data.iloc[i]
            previous = self.data.iloc[i-1]
            
            # Doji pattern
            if abs(current['open'] - current['close']) <= (current['high'] - current['low']) * 0.1:
                patterns['doji'].append(i)
            
            # Hammer pattern
            body_size = abs(current['close'] - current['open'])
            lower_shadow = min(current['open'], current['close']) - current['low']
            upper_shadow = current['high'] - max(current['open'], current['close'])
            
            if lower_shadow > body_size * 2 and upper_shadow < body_size * 0.5:
                patterns['hammer'].append(i)
            
            # Shooting star pattern
            if upper_shadow > body_size * 2 and lower_shadow < body_size * 0.5:
                patterns['shooting_star'].append(i)
            
            # Engulfing patterns
            if (previous['close'] < previous['open'] and current['close'] > current['open'] and
                current['open'] < previous['close'] and current['close'] > previous['open']):
                patterns['engulfing_bullish'].append(i)
            
            if (previous['close'] > previous['open'] and current['close'] < current['open'] and
                current['open'] > previous['close'] and current['close'] < previous['open']):
                patterns['engulfing_bearish'].append(i)
        
        return patterns
    
    def plot_candlestick_chart(self, inst_id: str = None, show_volume: bool = True, 
                              show_indicators: bool = True, height: int = 800):
        """Create interactive candlestick chart with Plotly"""
        data_to_plot = self.data
        if inst_id:
            data_to_plot = self.data[self.data.get('instId') == inst_id] if 'instId' in self.data.columns else self.data
        
        if data_to_plot.empty:
            print("No data to plot")
            return
        
        # Create subplots
        rows = 3 if show_volume and show_indicators else (2 if show_volume or show_indicators else 1)
        subplot_titles = ['Price', 'Volume', 'Indicators'] if rows == 3 else (['Price', 'Volume'] if show_volume else ['Price', 'Indicators'])
        
        fig = make_subplots(
            rows=rows, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=subplot_titles,
            row_heights=[0.6, 0.2, 0.2] if rows == 3 else ([0.7, 0.3] if rows == 2 else [1.0])
        )
        
        # Candlestick chart
        fig.add_trace(
            go.Candlestick(
                x=data_to_plot['timestamp'],
                open=data_to_plot['open'],
                high=data_to_plot['high'],
                low=data_to_plot['low'],
                close=data_to_plot['close'],
                name='Price'
            ),
            row=1, col=1
        )
        
        # Add moving averages if available
        if show_indicators and 'MA20' in data_to_plot.columns:
            fig.add_trace(
                go.Scatter(
                    x=data_to_plot['timestamp'],
                    y=data_to_plot['MA20'],
                    mode='lines',
                    name='MA20',
                    line=dict(color='orange', width=1)
                ),
                row=1, col=1
            )
        
        # Volume chart
        if show_volume:
            colors = ['red' if close < open else 'green' 
                     for close, open in zip(data_to_plot['close'], data_to_plot['open'])]
            
            fig.add_trace(
                go.Bar(
                    x=data_to_plot['timestamp'],
                    y=data_to_plot['volume'],
                    name='Volume',
                    marker_color=colors,
                    opacity=0.6
                ),
                row=2 if not show_indicators else 2, col=1
            )
        
        # RSI chart
        if show_indicators and 'RSI' in data_to_plot.columns:
            row_idx = 3 if show_volume else 2
            fig.add_trace(
                go.Scatter(
                    x=data_to_plot['timestamp'],
                    y=data_to_plot['RSI'],
                    mode='lines',
                    name='RSI',
                    line=dict(color='purple')
                ),
                row=row_idx, col=1
            )
            
            # Add RSI levels
            fig.add_hline(y=70, line_dash="dash", line_color="red", opacity=0.5, row=row_idx, col=1)
            fig.add_hline(y=30, line_dash="dash", line_color="green", opacity=0.5, row=row_idx, col=1)
        
        # Update layout
        title = f"Candlestick Chart - {inst_id}" if inst_id else "Candlestick Chart"
        fig.update_layout(
            title=title,
            height=height,
            xaxis_rangeslider_visible=False,
            showlegend=True
        )
        
        fig.show()
        
        return fig

In [ ]:
# ...existing code...

class TradeDataAnalyzer:
    """Enhanced trade data analyzer with candlestick conversion"""
    
    def __init__(self, data: pd.DataFrame):
        self.data = data
    
    def convert_to_candlesticks(self, timeframe: str = '1min', inst_id: str = None) -> pd.DataFrame:
        """Convert trade data to candlestick format"""
        data_to_convert = self.data
        if inst_id:
            data_to_convert = self.data[self.data['instId'] == inst_id]
        
        if data_to_convert.empty:
            return pd.DataFrame()
        
        # Set timeframe rules
        if timeframe == '1min':
            freq = '1T'
        elif timeframe == '5min':
            freq = '5T'
        elif timeframe == '15min':
            freq = '15T'
        elif timeframe == '1hour':
            freq = '1H'
        elif timeframe == '1day':
            freq = '1D'
        else:
            freq = '1T'
        
        # Group by timeframe
        data_to_convert = data_to_convert.set_index('datetime')
        
        ohlcv = data_to_convert.groupby([pd.Grouper(freq=freq), 'instId']).agg({
            'price': ['first', 'max', 'min', 'last'],
            'size': 'sum',
            'tradeId': 'count'
        }).reset_index()
        
        # Flatten column names
        ohlcv.columns = ['timestamp', 'instId', 'open', 'high', 'low', 'close', 'volume', 'trade_count']
        
        # Remove empty periods
        ohlcv = ohlcv.dropna()
        
        return ohlcv.sort_values('timestamp').reset_index(drop=True)
    
    # ...existing code...
    
    def plot_price_vs_volume(self, inst_id: str = None):
        """Plot price vs volume correlation"""
        data_to_plot = self.data if inst_id is None else self.data[self.data['instId'] == inst_id]
        
        if data_to_plot.empty:
            print("No data to plot")
            return
        
        # Convert to candlesticks first
        candlestick_data = self.convert_to_candlesticks('5min', inst_id)
        
        if candlestick_data.empty:
            print("No candlestick data to plot")
            return
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), height_ratios=[2, 1])
        
        # Price chart
        ax1.plot(candlestick_data['timestamp'], candlestick_data['close'], 
                label='Close Price', color='blue', linewidth=1)
        ax1.fill_between(candlestick_data['timestamp'], 
                        candlestick_data['low'], candlestick_data['high'], 
                        alpha=0.3, color='lightblue', label='Price Range')
        ax1.set_ylabel('Price')
        ax1.set_title(f'Price and Volume Analysis - {inst_id if inst_id else "All Instruments"}')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Volume chart
        colors = ['red' if close < open else 'green' 
                 for close, open in zip(candlestick_data['close'], candlestick_data['open'])]
        ax2.bar(candlestick_data['timestamp'], candlestick_data['volume'], 
                color=colors, alpha=0.7, width=0.8)
        ax2.set_ylabel('Volume')
        ax2.set_xlabel('Time')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # ...existing code...

In [ ]:
# ...existing code...

class TradeDatabase:
    """Enhanced database operations with candlestick data support"""
    
    def __init__(self, db_path: str = "trade_data.db"):
        self.db_path = db_path
        self.init_database()
    
    def init_database(self):
        """Initialize database with both trade and candlestick tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS trades (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                instType TEXT,
                instId TEXT,
                timestamp INTEGER,
                datetime TEXT,
                price REAL,
                size REAL,
                side TEXT,
                tradeId TEXT,
                received_at TEXT,
                UNIQUE(instId, tradeId, timestamp)
            )
        ''')
        
        # Create indexes for better query performance
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_instId ON trades(instId)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_timestamp ON trades(timestamp)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_datetime ON trades(datetime)')
        
        # Create candlestick data table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS candlesticks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                instId TEXT,
                timeframe TEXT,
                timestamp TEXT,
                open REAL,
                high REAL,
                low REAL,
                close REAL,
                volume REAL,
                trade_count INTEGER,
                created_at TEXT,
                UNIQUE(instId, timeframe, timestamp)
            )
        ''')
        
        # Create indexes for candlestick data
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_candlestick_instId ON candlesticks(instId)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_candlestick_timeframe ON candlesticks(timeframe)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_candlestick_timestamp ON candlesticks(timestamp)')
        
        conn.commit()
        conn.close()
    
    def save_trades(self, trades_df: pd.DataFrame):
        """Save trade data to database"""
        if trades_df.empty:
            return
            
        conn = sqlite3.connect(self.db_path)
        
        try:
            trades_df.to_sql('trades', conn, if_exists='append', index=False)
            logger.info(f"Saved {len(trades_df)} trades to database")
        except sqlite3.IntegrityError:
            # Handle duplicate entries
            for _, row in trades_df.iterrows():
                try:
                    row.to_frame().T.to_sql('trades', conn, if_exists='append', index=False)
                except sqlite3.IntegrityError:
                    continue  # Skip duplicates
        finally:
            conn.close()
    
    def save_candlesticks(self, candlestick_df: pd.DataFrame, timeframe: str = '1min'):
        """Save candlestick data to database"""
        if candlestick_df.empty:
            return
        
        # Add timeframe and created_at columns
        candlestick_df = candlestick_df.copy()
        candlestick_df['timeframe'] = timeframe
        candlestick_df['created_at'] = datetime.now().isoformat()
        
        conn = sqlite3.connect(self.db_path)
        
        try:
            candlestick_df.to_sql('candlesticks', conn, if_exists='append', index=False)
            logger.info(f"Saved {len(candlestick_df)} candlesticks to database")
        except sqlite3.IntegrityError:
            # Handle duplicates
            for _, row in candlestick_df.iterrows():
                try:
                    row.to_frame().T.to_sql('candlesticks', conn, if_exists='append', index=False)
                except sqlite3.IntegrityError:
                    continue
        finally:
            conn.close()
    
    def load_trades(self, inst_id: str = None, start_time: str = None, end_time: str = None) -> pd.DataFrame:
        """Load trade data from database"""
        conn = sqlite3.connect(self.db_path)
        
        query = "SELECT * FROM trades WHERE 1=1"
        params = []
        
        if inst_id:
            query += " AND instId = ?"
            params.append(inst_id)
            
        if start_time:
            query += " AND datetime >= ?"
            params.append(start_time)
            
        if end_time:
            query += " AND datetime <= ?"
            params.append(end_time)
            
        query += " ORDER BY timestamp"
        
        df = pd.read_sql_query(query, conn, params=params)
        conn.close()
        
        if not df.empty:
            df['datetime'] = pd.to_datetime(df['datetime'])
        
        return df
    
    def load_candlesticks(self, inst_id: str = None, timeframe: str = '1min', 
                         start_time: str = None, end_time: str = None) -> pd.DataFrame:
        """Load candlestick data from database"""
        conn = sqlite3.connect(self.db_path)
        
        query = "SELECT * FROM candlesticks WHERE 1=1"
        params = []
        
        if inst_id:
            query += " AND instId = ?"
            params.append(inst_id)
            
        if timeframe:
            query += " AND timeframe = ?"
            params.append(timeframe)
            
        if start_time:
            query += " AND timestamp >= ?"
            params.append(start_time)
            
        if end_time:
            query += " AND timestamp <= ?"
            params.append(end_time)
            
        query += " ORDER BY timestamp"
        
        df = pd.read_sql_query(query, conn, params=params)
        conn.close()
        
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        return df
    
    # ...existing code...

## Real-time Candlestick Analysis

Examples of real-time candlestick generation and technical analysis.

In [ ]:
def candlestick_analysis_example():
    """Comprehensive example of candlestick analysis with real-time data"""
    
    # Create enhanced client
    client = TradeChannelClient()
    db = TradeDatabase("candlestick_data.db")
    
    print("Starting candlestick analysis...")
    client.connect()
    
    if not client.is_connected:
        print("Failed to connect")
        return
    
    # Subscribe to instruments
    instruments = ["BTCUSDT", "ETHUSDT"]
    
    for inst_id in instruments:
        print(f"Subscribing to {inst_id}...")
        client.subscribe("USDT-FUTURES", inst_id)
        time.sleep(1)
    
    # Collect data for analysis
    print("Collecting data for candlestick analysis (60 seconds)...")
    time.sleep(60)
    
    # Generate candlestick data
    for inst_id in instruments:
        candlestick_data = client.get_candlestick_data(inst_id)
        
        if not candlestick_data.empty:
            print(f"\n{inst_id} Candlestick Data:")
            print(f"Total candles: {len(candlestick_data)}")
            print(candlestick_data.tail())
            
            # Technical analysis
            analyzer = CandlestickAnalyzer(candlestick_data)
            analyzer.add_technical_indicators()
            
            # Display current indicators
            latest = analyzer.data.iloc[-1]
            print(f"\nLatest indicators for {inst_id}:")
            print(f"  Close: ${latest['close']:.2f}")
            print(f"  RSI: {latest.get('RSI', 'N/A'):.2f}" if 'RSI' in analyzer.data.columns else "  RSI: N/A")
            print(f"  MA20: ${latest.get('MA20', 'N/A'):.2f}" if 'MA20' in analyzer.data.columns else "  MA20: N/A")
            
            # Detect patterns
            patterns = analyzer.detect_patterns()
            print(f"  Detected patterns: {len([p for p in patterns.values() if p])}")
            
            # Save to database
            db.save_candlesticks(candlestick_data, '1min')
            
            # Create visualization
            print(f"Creating candlestick chart for {inst_id}...")
            analyzer.plot_candlestick_chart(inst_id, show_volume=True, show_indicators=True)
    
    client.close()
    print("Analysis completed")

# Uncomment to run candlestick analysis
# candlestick_analysis_example()

In [ ]:
def technical_analysis_dashboard():
    """Create a comprehensive technical analysis dashboard"""
    
    # Load historical data
    db = TradeDatabase("candlestick_data.db")
    
    # Get data for analysis
    inst_id = "BTCUSDT"
    end_time = datetime.now()
    start_time = end_time - timedelta(hours=2)
    
    candlestick_data = db.load_candlesticks(
        inst_id=inst_id,
        timeframe='1min',
        start_time=start_time.isoformat(),
        end_time=end_time.isoformat()
    )
    
    if candlestick_data.empty:
        print("No historical candlestick data found")
        return
    
    print(f"Analyzing {len(candlestick_data)} candlesticks for {inst_id}")
    
    # Technical analysis
    analyzer = CandlestickAnalyzer(candlestick_data)
    analyzer.add_technical_indicators()
    
    # Create comprehensive dashboard
    fig = make_subplots(
        rows=4, cols=2,
        subplot_titles=[
            'Candlestick Chart', 'Volume Analysis',
            'RSI Indicator', 'MACD Indicator', 
            'Bollinger Bands', 'Price Distribution',
            'Trade Count', 'Volume Profile'
        ],
        specs=[
            [{"colspan": 2}, None],
            [{"secondary_y": False}, {"secondary_y": False}],
            [{"secondary_y": False}, {"secondary_y": False}],
            [{"secondary_y": False}, {"secondary_y": False}]
        ],
        vertical_spacing=0.08,
        horizontal_spacing=0.05
    )
    
    # Main candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=analyzer.data['timestamp'],
            open=analyzer.data['open'],
            high=analyzer.data['high'],
            low=analyzer.data['low'],
            close=analyzer.data['close'],
            name='Price'
        ),
        row=1, col=1
    )
    
    # Volume
    colors = ['red' if close < open else 'green' 
             for close, open in zip(analyzer.data['close'], analyzer.data['open'])]
    fig.add_trace(
        go.Bar(x=analyzer.data['timestamp'], y=analyzer.data['volume'], 
               name='Volume', marker_color=colors, opacity=0.6),
        row=2, col=1
    )
    
    # RSI
    if 'RSI' in analyzer.data.columns:
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['RSI'], 
                      name='RSI', line=dict(color='purple')),
            row=2, col=2
        )
        fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=2)
        fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=2)
    
    # MACD
    if 'MACD' in analyzer.data.columns:
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['MACD'], 
                      name='MACD', line=dict(color='blue')),
            row=3, col=1
        )
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['MACD_signal'], 
                      name='Signal', line=dict(color='red')),
            row=3, col=1
        )
    
    # Bollinger Bands
    if 'BB_upper' in analyzer.data.columns:
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['BB_upper'], 
                      name='BB Upper', line=dict(color='gray', dash='dash')),
            row=3, col=2
        )
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['close'], 
                      name='Close', line=dict(color='black')),
            row=3, col=2
        )
        fig.add_trace(
            go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['BB_lower'], 
                      name='BB Lower', line=dict(color='gray', dash='dash')),
            row=3, col=2
        )
    
    # Price distribution
    fig.add_trace(
        go.Histogram(y=analyzer.data['close'], name='Price Dist', nbinsy=30),
        row=4, col=1
    )
    
    # Trade count over time
    fig.add_trace(
        go.Scatter(x=analyzer.data['timestamp'], y=analyzer.data['trade_count'], 
                  name='Trade Count', mode='lines+markers'),
        row=4, col=2
    )
    
    # Update layout
    fig.update_layout(
        title=f"Technical Analysis Dashboard - {inst_id}",
        height=1200,
        showlegend=False
    )
    
    fig.show()
    
    # Print summary statistics
    latest = analyzer.data.iloc[-1]
    print(f"\nCurrent Market Summary for {inst_id}:")
    print(f"  Price: ${latest['close']:.2f}")
    print(f"  24h Change: {((latest['close'] - analyzer.data.iloc[0]['close']) / analyzer.data.iloc[0]['close'] * 100):.2f}%")
    print(f"  Volume: {latest['volume']:.4f}")
    print(f"  RSI: {latest.get('RSI', 'N/A'):.2f}" if 'RSI' in analyzer.data.columns else "  RSI: N/A")
    
    # Pattern detection
    patterns = analyzer.detect_patterns()
    active_patterns = [name for name, indices in patterns.items() if indices]
    print(f"  Active Patterns: {', '.join(active_patterns) if active_patterns else 'None'}")

# Uncomment to run technical analysis dashboard
# technical_analysis_dashboard()

In [ ]:
# Example: Basic usage of Trade Channel client for SPOT
def basic_example():
    """Basic example of connecting and subscribing to SPOT trade data"""
    
    # Create client instance
    client = TradeChannelClient()
    
    # Connect to WebSocket
    print("Connecting to WebSocket...")
    client.connect()
    
    if not client.is_connected:
        print("Failed to connect")
        return
    
    # Subscribe to BTCUSDT SPOT trades
    print("Subscribing to BTCUSDT SPOT trades...")
    client.subscribe("SPOT", "BTCUSDT")  # Thay đổi thành SPOT
    
    # Collect data for 30 seconds
    print("Collecting data for 30 seconds...")
    time.sleep(30)
    
    # Get collected data
    trade_df = client.get_trade_data()
    print(f"Collected {len(trade_df)} SPOT trades")
    
    if not trade_df.empty:
        print("\nSample SPOT data:")
        print(trade_df.head())
        
        # Basic analysis
        analyzer = TradeDataAnalyzer(trade_df)
        stats = analyzer.get_basic_stats()
        print(f"\nBasic SPOT statistics:")
        for inst_id, stat in stats.items():
            print(f"{inst_id}: {stat}")
    
    # Close connection
    client.close()
    print("Connection closed")

# Uncomment to run the basic example
# basic_example()

In [ ]:
# Example: Advanced usage with multiple instruments and database storage
def advanced_example():
    """Advanced example with multiple subscriptions and data persistence"""
    
    # Create client and database instances
    client = TradeChannelClient()
    db = TradeDatabase("trade_data.db")
    
    # Connect to WebSocket
    print("Connecting to WebSocket...")
    client.connect()
    
    if not client.is_connected:
        print("Failed to connect")
        return
    
    # Subscribe to multiple instruments
    instruments = [
        ("USDT-FUTURES", "BTCUSDT"),
        ("USDT-FUTURES", "ETHUSDT"),
        ("USDT-FUTURES", "ADAUSDT")
    ]
    
    for inst_type, inst_id in instruments:
        print(f"Subscribing to {inst_id}...")
        client.subscribe(inst_type, inst_id)
        time.sleep(1)  # Small delay between subscriptions
    
    # Collect data for 60 seconds
    print("Collecting data for 60 seconds...")
    start_time = time.time()
    
    while time.time() - start_time < 60:
        time.sleep(5)  # Save data every 5 seconds
        
        # Get current data and save to database
        current_data = client.get_trade_data()
        if not current_data.empty:
            # Only save new data (simple approach - in production, implement proper deduplication)
            db.save_trades(current_data.tail(50))  # Save last 50 trades
    
    # Final data collection
    final_data = client.get_trade_data()
    print(f"Collected {len(final_data)} total trades")
    
    # Analysis
    if not final_data.empty:
        analyzer = TradeDataAnalyzer(final_data)
        
        # Show statistics
        stats = analyzer.get_basic_stats()
        for inst_id, stat in stats.items():
            print(f"\n{inst_id} Statistics:")
            print(f"  Total trades: {stat['total_trades']}")
            print(f"  Average price: ${stat['avg_price']:.2f}")
            print(f"  Price range: ${stat['price_range']['min']:.2f} - ${stat['price_range']['max']:.2f}")
            print(f"  Total volume: {stat['total_volume']:.4f}")
            print(f"  Buy/Sell ratio: {stat['buy_trades']}/{stat['sell_trades']}")
        
        # Generate plots
        print("\nGenerating analysis plots...")
        analyzer.plot_price_distribution()
        analyzer.plot_trade_timeline()
        
        # Individual instrument analysis
        for inst_type, inst_id in instruments:
            if inst_id in final_data['instId'].values:
                print(f"\nAnalyzing {inst_id}...")
                analyzer.plot_price_distribution(inst_id)
                analyzer.plot_trade_timeline(inst_id)
                
                # Volume profile
                volume_profile = analyzer.get_volume_profile(inst_id)
                if not volume_profile.empty:
                    print(f"Volume profile for {inst_id}:")
                    print(volume_profile.head(10))
    
    # Close connection
    client.close()
    print("Connection closed")

# Uncomment to run the advanced example
# advanced_example()

In [ ]:
# Example: Real-time monitoring with alerts
class TradeMonitor:
    """Real-time trade monitoring with price alerts"""
    
    def __init__(self, client: TradeChannelClient):
        self.client = client
        self.price_alerts = {}  # {inst_id: {'high': price, 'low': price}}
        self.volume_alerts = {}  # {inst_id: threshold}
        
    def set_price_alert(self, inst_id: str, high_price: float = None, low_price: float = None):
        """Set price alerts for an instrument"""
        self.price_alerts[inst_id] = {
            'high': high_price,
            'low': low_price
        }
        
    def set_volume_alert(self, inst_id: str, volume_threshold: float):
        """Set volume alert for an instrument"""
        self.volume_alerts[inst_id] = volume_threshold
        
    def check_alerts(self):
        """Check for triggered alerts in recent trade data"""
        recent_data = self.client.get_trade_data()
        if recent_data.empty:
            return
            
        # Check last 10 trades for each instrument
        for inst_id in recent_data['instId'].unique():
            inst_data = recent_data[recent_data['instId'] == inst_id].tail(10)
            
            # Price alerts
            if inst_id in self.price_alerts:
                alerts = self.price_alerts[inst_id]
                
                if alerts['high'] and inst_data['price'].max() >= alerts['high']:
                    print(f"🚨 HIGH PRICE ALERT: {inst_id} reached ${inst_data['price'].max():.2f}")
                    
                if alerts['low'] and inst_data['price'].min() <= alerts['low']:
                    print(f"🚨 LOW PRICE ALERT: {inst_id} dropped to ${inst_data['price'].min():.2f}")
            
            # Volume alerts
            if inst_id in self.volume_alerts:
                large_trades = inst_data[inst_data['size'] >= self.volume_alerts[inst_id]]
                if not large_trades.empty:
                    for _, trade in large_trades.iterrows():
                        print(f"📊 VOLUME ALERT: {inst_id} large {trade['side']} of {trade['size']:.4f} at ${trade['price']:.2f}")

def monitoring_example():
    """Example of real-time monitoring with alerts"""
    
    # Create client and monitor
    client = TradeChannelClient()
    monitor = TradeMonitor(client)
    
    # Connect and subscribe
    print("Starting real-time monitoring...")
    client.connect()
    
    if not client.is_connected:
        print("Failed to connect")
        return
    
    # Subscribe to BTCUSDT
    client.subscribe("USDT-FUTURES", "BTCUSDT")
    
    # Set up alerts (example thresholds - adjust based on current market prices)
    monitor.set_price_alert("BTCUSDT", high_price=50000, low_price=40000)
    monitor.set_volume_alert("BTCUSDT", volume_threshold=1.0)
    
    print("Monitoring for 2 minutes...")
    start_time = time.time()
    
    while time.time() - start_time < 120:  # Monitor for 2 minutes
        time.sleep(5)  # Check every 5 seconds
        monitor.check_alerts()
        
        # Show current data summary
        current_data = client.get_trade_data()
        if not current_data.empty:
            recent_trades = current_data.tail(5)
            print(f"Recent trades: {len(current_data)} total, last price: ${recent_trades['price'].iloc[-1]:.2f}")
    
    client.close()
    print("Monitoring stopped")

# Uncomment to run the monitoring example
# monitoring_example()

## Advanced Analysis Features

Additional tools for deep market analysis and pattern recognition.

In [ ]:
def market_microstructure_analysis():
    """Analyze market microstructure from trade data"""
    
    client = TradeChannelClient()
    client.connect()
    
    if not client.is_connected:
        print("Failed to connect")
        return
    
    # Subscribe to high-frequency data
    client.subscribe("USDT-FUTURES", "BTCUSDT")
    
    print("Collecting high-frequency data for microstructure analysis...")
    time.sleep(120)  # 2 minutes of data
    
    # Get trade data
    trade_data = client.get_trade_data()
    
    if trade_data.empty:
        print("No trade data collected")
        client.close()
        return
    
    print(f"Analyzing {len(trade_data)} trades")
    
    # Order flow analysis
    buy_trades = trade_data[trade_data['side'] == 'buy']
    sell_trades = trade_data[trade_data['side'] == 'sell']
    
    print(f"\nOrder Flow Analysis:")
    print(f"  Buy trades: {len(buy_trades)} ({len(buy_trades)/len(trade_data)*100:.1f}%)")
    print(f"  Sell trades: {len(sell_trades)} ({len(sell_trades)/len(trade_data)*100:.1f}%)")
    print(f"  Buy volume: {buy_trades['size'].sum():.4f}")
    print(f"  Sell volume: {sell_trades['size'].sum():.4f}")
    
    # Price impact analysis
    trade_data['price_change'] = trade_data['price'].diff()
    trade_data['size_category'] = pd.cut(trade_data['size'], 
                                        bins=[0, 0.01, 0.1, 1.0, float('inf')], 
                                        labels=['Small', 'Medium', 'Large', 'Whale'])
    
    impact_analysis = trade_data.groupby(['side', 'size_category']).agg({
        'price_change': ['mean', 'std', 'count'],
        'size': ['mean', 'sum']
    }).round(4)
    
    print(f"\nPrice Impact by Trade Size:")
    print(impact_analysis)
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Trade size distribution
    axes[0,0].hist([buy_trades['size'], sell_trades['size']], 
                   bins=50, alpha=0.7, label=['Buy', 'Sell'], color=['green', 'red'])
    axes[0,0].set_title('Trade Size Distribution')
    axes[0,0].set_xlabel('Trade Size')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].legend()
    axes[0,0].set_yscale('log')
    
    # Price vs Volume scatter
    colors = ['green' if side == 'buy' else 'red' for side in trade_data['side']]
    axes[0,1].scatter(trade_data['size'], trade_data['price'], 
                      c=colors, alpha=0.6, s=10)
    axes[0,1].set_title('Price vs Trade Size')
    axes[0,1].set_xlabel('Trade Size')
    axes[0,1].set_ylabel('Price')
    
    # Time series of trades
    trade_data['cumulative_volume'] = trade_data['size'].cumsum()
    axes[1,0].plot(trade_data['datetime'], trade_data['cumulative_volume'])
    axes[1,0].set_title('Cumulative Volume Over Time')
    axes[1,0].set_xlabel('Time')
    axes[1,0].set_ylabel('Cumulative Volume')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Buy/Sell pressure over time
    trade_data['minute'] = trade_data['datetime'].dt.floor('1min')
    minute_summary = trade_data.groupby(['minute', 'side'])['size'].sum().unstack(fill_value=0)
    
    if 'buy' in minute_summary.columns and 'sell' in minute_summary.columns:
        minute_summary['net_flow'] = minute_summary['buy'] - minute_summary['sell']
        axes[1,1].bar(minute_summary.index, minute_summary['net_flow'], 
                      color=['green' if x > 0 else 'red' for x in minute_summary['net_flow']])
        axes[1,1].set_title('Net Order Flow (Buy - Sell)')
        axes[1,1].set_xlabel('Time')
        axes[1,1].set_ylabel('Net Volume')
        axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    client.close()
    print("Microstructure analysis completed")

# Uncomment to run microstructure analysis
# market_microstructure_analysis()

In [ ]:
def export_data_to_csv(client: TradeChannelClient, filename: str = None):
    """Export collected trade data to CSV file"""
    
    if filename is None:
        filename = f"trade_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    trade_data = client.get_trade_data()
    
    if trade_data.empty:
        print("No data to export")
        return None
    
    trade_data.to_csv(filename, index=False)
    print(f"Exported {len(trade_data)} trades to {filename}")
    return filename

def load_and_analyze_historical_data(db: TradeDatabase, inst_id: str, hours: int = 24):
    """Load and analyze historical data from database"""
    
    # Calculate time range
    end_time = datetime.now()
    start_time = end_time - pd.Timedelta(hours=hours)
    
    # Load data
    historical_data = db.load_trades(
        inst_id=inst_id,
        start_time=start_time.isoformat(),
        end_time=end_time.isoformat()
    )
    
    if historical_data.empty:
        print(f"No historical data found for {inst_id}")
        return None
    
    print(f"Loaded {len(historical_data)} historical trades for {inst_id}")
    
    # Analyze
    analyzer = TradeDataAnalyzer(historical_data)
    stats = analyzer.get_basic_stats()
    
    print(f"Historical analysis for {inst_id}:")
    if inst_id in stats:
        stat = stats[inst_id]
        print(f"  Time period: {stat['time_range']['start']} to {stat['time_range']['end']}")
        print(f"  Total trades: {stat['total_trades']}")
        print(f"  Average price: ${stat['avg_price']:.2f}")
        print(f"  Price volatility: {historical_data['price'].std():.2f}")
        print(f"  Total volume: {stat['total_volume']:.4f}")
    
    # Generate plots
    analyzer.plot_price_distribution(inst_id)
    analyzer.plot_trade_timeline(inst_id)
    
    return historical_data

# Example usage of export and analysis functions
def export_and_analysis_example():
    """Example of data export and historical analysis"""
    
    # Create database instance
    db = TradeDatabase("trade_data.db")
    
    # Analyze historical data (if available)
    historical_data = load_and_analyze_historical_data(db, "BTCUSDT", hours=24)
    
    if historical_data is not None:
        # Export to CSV
        csv_filename = f"BTCUSDT_historical_{datetime.now().strftime('%Y%m%d')}.csv"
        historical_data.to_csv(csv_filename, index=False)
        print(f"Historical data exported to {csv_filename}")

# Uncomment to run export and analysis example
# export_and_analysis_example()

## Configuration and Trading Strategies

Configuration settings and example trading strategies based on candlestick patterns.

In [ ]:
# Enhanced configuration with candlestick settings
CANDLESTICK_CONFIG = {
    'timeframes': ['1min', '5min', '15min', '1hour', '4hour', '1day'],
    'default_timeframe': '1min',
    'indicators': {
        'sma_periods': [5, 10, 20, 50],
        'ema_periods': [12, 26],
        'rsi_period': 14,
        'macd_fast': 12,
        'macd_slow': 26,
        'macd_signal': 9,
        'bollinger_period': 20,
        'bollinger_std': 2
    },
    'pattern_detection': {
        'doji_threshold': 0.1,
        'hammer_ratio': 2.0,
        'engulfing_min_body': 0.6
    }
}

# Trading strategy based on candlestick patterns
class CandlestickStrategy:
    """Simple trading strategy based on candlestick patterns and indicators"""
    
    def __init__(self, analyzer: CandlestickAnalyzer):
        self.analyzer = analyzer
        self.signals = []
    
    def generate_signals(self):
        """Generate buy/sell signals based on patterns and indicators"""
        if self.analyzer.data.empty or len(self.analyzer.data) < 20:
            return []
        
        patterns = self.analyzer.detect_patterns()
        signals = []
        
        for i in range(20, len(self.analyzer.data)):  # Need enough data for indicators
            current = self.analyzer.data.iloc[i]
            signal = {
                'timestamp': current['timestamp'],
                'price': current['close'],
                'signal': 'hold',
                'confidence': 0.0,
                'reasons': []
            }
            
            # RSI-based signals
            if 'RSI' in self.analyzer.data.columns:
                rsi = current['RSI']
                if rsi < 30:
                    signal['signal'] = 'buy'
                    signal['confidence'] += 0.3
                    signal['reasons'].append('RSI oversold')
                elif rsi > 70:
                    signal['signal'] = 'sell'
                    signal['confidence'] += 0.3
                    signal['reasons'].append('RSI overbought')
            
            # Moving average crossover
            if 'MA5' in self.analyzer.data.columns and 'MA20' in self.analyzer.data.columns:
                ma5_current = current['MA5']
                ma20_current = current['MA20']
                ma5_prev = self.analyzer.data.iloc[i-1]['MA5']
                ma20_prev = self.analyzer.data.iloc[i-1]['MA20']
                
                # Golden cross (bullish)
                if ma5_prev <= ma20_prev and ma5_current > ma20_current:
                    signal['signal'] = 'buy'
                    signal['confidence'] += 0.4
                    signal['reasons'].append('MA golden cross')
                # Death cross (bearish)
                elif ma5_prev >= ma20_prev and ma5_current < ma20_current:
                    signal['signal'] = 'sell'
                    signal['confidence'] += 0.4
                    signal['reasons'].append('MA death cross')
            
            # Pattern-based signals
            if i in patterns.get('hammer', []):
                signal['signal'] = 'buy'
                signal['confidence'] += 0.2
                signal['reasons'].append('Hammer pattern')
            
            if i in patterns.get('shooting_star', []):
                signal['signal'] = 'sell'
                signal['confidence'] += 0.2
                signal['reasons'].append('Shooting star pattern')
            
            if i in patterns.get('engulfing_bullish', []):
                signal['signal'] = 'buy'
                signal['confidence'] += 0.3
                signal['reasons'].append('Bullish engulfing')
            
            if i in patterns.get('engulfing_bearish', []):
                signal['signal'] = 'sell'
                signal['confidence'] += 0.3
                signal['reasons'].append('Bearish engulfing')
            
            # Only keep signals with reasonable confidence
            if signal['confidence'] >= 0.3 and signal['signal'] != 'hold':
                signals.append(signal)
        
        self.signals = signals
        return signals
    
    def backtest_strategy(self, initial_balance: float = 1000.0):
        """Simple backtest of the strategy"""
        if not self.signals:
            self.generate_signals()
        
        balance = initial_balance
        position = 0  # 0 = no position, 1 = long, -1 = short
        trades = []
        
        for signal in self.signals:
            if signal['signal'] == 'buy' and position <= 0:
                if position < 0:
                    # Close short position
                    profit = (trades[-1]['entry_price'] - signal['price']) * abs(trades[-1]['quantity'])
                    balance += profit
                    trades[-1]['exit_price'] = signal['price']
                    trades[-1]['profit'] = profit
                
                # Open long position
                quantity = balance * 0.95 / signal['price']  # Use 95% of balance
                trades.append({
                    'type': 'long',
                    'entry_time': signal['timestamp'],
                    'entry_price': signal['price'],
                    'quantity': quantity,
                    'confidence': signal['confidence'],
                    'reasons': signal['reasons']
                })
                position = 1
                
            elif signal['signal'] == 'sell' and position >= 0:
                if position > 0:
                    # Close long position
                    profit = (signal['price'] - trades[-1]['entry_price']) * trades[-1]['quantity']
                    balance += profit
                    trades[-1]['exit_price'] = signal['price']
                    trades[-1]['profit'] = profit
                
                # Open short position (simplified)
                quantity = balance * 0.95 / signal['price']
                trades.append({
                    'type': 'short',
                    'entry_time': signal['timestamp'],
                    'entry_price': signal['price'],
                    'quantity': quantity,
                    'confidence': signal['confidence'],
                    'reasons': signal['reasons']
                })
                position = -1
        
        # Close any remaining position
        if trades and 'exit_price' not in trades[-1]:
            last_price = self.analyzer.data.iloc[-1]['close']
            if trades[-1]['type'] == 'long':
                profit = (last_price - trades[-1]['entry_price']) * trades[-1]['quantity']
            else:
                profit = (trades[-1]['entry_price'] - last_price) * trades[-1]['quantity']
            
            balance += profit
            trades[-1]['exit_price'] = last_price
            trades[-1]['profit'] = profit
        
        return {
            'initial_balance': initial_balance,
            'final_balance': balance,
            'total_return': (balance - initial_balance) / initial_balance * 100,
            'total_trades': len(trades),
            'profitable_trades': len([t for t in trades if t.get('profit', 0) > 0]),
            'trades': trades
        }

def strategy_example():
    """Example of using candlestick strategy"""
    
    # Load some historical data
    db = TradeDatabase("candlestick_data.db")
    candlestick_data = db.load_candlesticks("BTCUSDT", "1min")
    
    if candlestick_data.empty:
        print("No historical data for strategy testing")
        return
    
    # Prepare analyzer
    analyzer = CandlestickAnalyzer(candlestick_data)
    analyzer.add_technical_indicators()
    
    # Create and test strategy
    strategy = CandlestickStrategy(analyzer)
    signals = strategy.generate_signals()
    
    print(f"Generated {len(signals)} trading signals")
    
    if signals:
        print("\nSample signals:")
        for signal in signals[:5]:
            print(f"  {signal['timestamp']}: {signal['signal'].upper()} at ${signal['price']:.2f}")
            print(f"    Confidence: {signal['confidence']:.2f}")
            print(f"    Reasons: {', '.join(signal['reasons'])}")
    
    # Backtest
    backtest_results = strategy.backtest_strategy(1000.0)
    
    print(f"\nBacktest Results:")
    print(f"  Initial Balance: ${backtest_results['initial_balance']:.2f}")
    print(f"  Final Balance: ${backtest_results['final_balance']:.2f}")
    print(f"  Total Return: {backtest_results['total_return']:.2f}%")
    print(f"  Total Trades: {backtest_results['total_trades']}")
    print(f"  Profitable Trades: {backtest_results['profitable_trades']}")
    
    if backtest_results['total_trades'] > 0:
        win_rate = backtest_results['profitable_trades'] / backtest_results['total_trades'] * 100
        print(f"  Win Rate: {win_rate:.1f}%")

# Uncomment to run strategy example
# strategy_example()

print("Candlestick Configuration:")
for key, value in CANDLESTICK_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Configuration settings - Updated for SPOT trading
WEBSOCKET_CONFIG = {
    'url': 'wss://ws-api.bingx.com/market',
    'ping_interval': 20,
    'ping_timeout': 10,
    'reconnect_attempts': 5,
    'reconnect_delay': 5
}

INSTRUMENT_TYPES = {
    'SPOT': 'SPOT',  # Thêm SPOT trading
    'USDT_FUTURES': 'USDT-FUTURES',
    'COIN_FUTURES': 'COIN-FUTURES', 
    'USDC_FUTURES': 'USDC-FUTURES',
    'SUSDT_FUTURES': 'USDT-M Futures Demo',
    'SCOIN_FUTURES': 'Coin-M Futures Demo',
    'SUSDC_FUTURES': 'USDC-M Futures Demo'
}

POPULAR_INSTRUMENTS = {
    'SPOT': [  # Thêm danh sách SPOT pairs
        'BTCUSDT', 'ETHUSDT', 'ADAUSDT', 'BNBUSDT', 'XRPUSDT',
        'SOLUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'MATICUSDT'
    ],
    'USDT-FUTURES': [
        'BTCUSDT', 'ETHUSDT', 'ADAUSDT', 'BNBUSDT', 'XRPUSDT',
        'SOLUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'MATICUSDT'
    ]
}

# Print available configurations
print("Available WebSocket Configuration:")
for key, value in WEBSOCKET_CONFIG.items():
    print(f"  {key}: {value}")

print("\nSupported Instrument Types:")
for key, value in INSTRUMENT_TYPES.items():
    print(f"  {key}: {value}")

print("\nPopular Trading Instruments:")
for inst_type, instruments in POPULAR_INSTRUMENTS.items():
    print(f"  {inst_type}: {', '.join(instruments[:5])}...")

## Summary

This enhanced notebook provides a comprehensive candlestick analysis framework for the Public Trade Channel WebSocket API. Key improvements include:

### New Features:
1. **Candlestick Generation**: Automatic conversion of trade data to OHLCV format
2. **Technical Indicators**: RSI, MACD, Bollinger Bands, Moving Averages
3. **Pattern Recognition**: Detection of classic candlestick patterns (Doji, Hammer, Engulfing, etc.)
4. **Interactive Charts**: Plotly-based candlestick charts with volume and indicators
5. **Trading Strategy**: Example strategy based on patterns and technical indicators
6. **Market Microstructure**: Order flow and price impact analysis
7. **Enhanced Database**: Separate storage for candlestick data

### Technical Analysis Tools:
- **Indicators**: Moving averages, RSI, MACD, Bollinger Bands
- **Patterns**: Doji, Hammer, Shooting Star, Engulfing patterns
- **Visualization**: Multi-panel charts with price, volume, and indicators
- **Strategy Backtesting**: Simple framework for testing trading strategies

### Usage Workflow:
1. **Data Collection**: Connect to WebSocket and collect real-time trades
2. **Candlestick Generation**: Automatic OHLCV aggregation by timeframe
3. **Technical Analysis**: Add indicators and detect patterns
4. **Visualization**: Create comprehensive charts and dashboards  
5. **Strategy Testing**: Backtest trading strategies on historical data
6. **Market Analysis**: Deep dive into market microstructure

### Next Steps:
- Implement more sophisticated trading strategies
- Add risk management and position sizing
- Create real-time alerts for pattern detection
- Integrate with trading APIs for automated execution
- Add more advanced technical indicators (Ichimoku, Elliott Wave, etc.)